# LBM-Suite2p-Python: Quickstart

LBM-Suite2p-Python is a volumetric wrapper around [Suite2p](https://suite2p.readthedocs.io/en/latest/) optimized for Light Beads Microscopy data.

## Supported Input Formats

| Format | Extension | Description |
|--------|-----------|-------------|
| TIFF | `.tif`, `.tiff` | Planar timeseries (T, Y, X) |
| Zarr | `.zarr` | Chunked array format |
| Suite2p Binary | `.bin` | Pre-converted binary with `ops.npy` |

**Note:** Raw ScanImage TIFFs must first be assembled using [mbo_utilities](https://millerbrainobservatory.github.io/mbo_utilities/assembly.html).

## Installation

```bash
pip install lbm-suite2p-python
# or from source:
pip install git+https://github.com/MillerBrainObservatory/LBM-Suite2p-Python.git
```

---

## 1. Prepare Input Files

### Option A: TIFF files (most common)

In [ ]:
# Path to assembled planar TIFFs

---

## 2. Configure Parameters

The `ops` dictionary controls all Suite2p and Cellpose parameters. You only need to specify values you want to override from defaults.

In [1]:
from pathlib import Path
import numpy as np
import mbo_utilities as mbo
import lbm_suite2p_python as lsp

ops = {
    "diameter": 4,                   # Expected cell diameter (pixels)
    "anatomical_only": 4,            # Cellpose mode: 0=off, 1=max, 2=mean, 3=enhanced, 4=max
    "accept_all_cells": True,
    "spatial_hp_cp": 3,
    "denoise": 1,
    "two_step_registration": 1,
}

# Process volume (returns list of ops files, one per plane)
ops_files = lsp.pipeline(
    input_data=Path(r"\\rbo-w1\D\W1_DATA\wsnyder\2025-10-16-Females-Shank-Wheel-316902\5"),
    save_path=Path(r"D:\output\wsnyder"),
    ops=ops,
    keep_raw=False,
    keep_reg=True,
    force_reg=False,
    force_detect=False,
)

print(f"Processed {len(ops_files)} planes")
print(f"Results saved to: {ops_files[0].parent.parent}")

Loading input data...
  Input: \\rbo-w1\D\W1_DATA\wsnyder\2025-10-16-Females-Shank-Wheel-316902\5


Counting frames:   0%|          | 0/49 [00:00<?, ?it/s]

  Loaded as: MboRawArray

Dataset info:
  Shape: (21880, 14, 448, 448)
  Frames: 21880
  Planes: 14
  Dimensions: 448 x 448
  Phase correction: True
  FFT subpixel: False
  Frame rate: 17.07 Hz
  ROIs: 2
Importing suite2p packages...

Processing plan:
  Planes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
  Output: D:\output\wsnyder

Processing plane 1/14
  Skipping plane01_stitched: already complete
Plotting results for 78 accepted / 13 rejected ROIs

Processing plane 2/14
  Skipping plane02_stitched: already complete
Plotting results for 90 accepted / 9 rejected ROIs

Processing plane 3/14
  Skipping plane03_stitched: already complete
Plotting results for 104 accepted / 19 rejected ROIs

Processing plane 4/14
  Skipping plane04_stitched: already complete
Plotting results for 103 accepted / 9 rejected ROIs

Processing plane 5/14
  Skipping plane05_stitched: already complete
Plotting results for 93 accepted / 12 rejected ROIs

Processing plane 6/14
  Skipping plane06_stitched: alrea

In [ ]:
from pprint import pprint

# List output files
output_files = sorted(save_path.rglob("*.*"))
print("Output files:")
pprint([f.relative_to(save_path) for f in output_files[:20]])

### Standard Suite2p Files

| File | Description |
|------|-------------|
| `ops.npy` | Parameters and processing metadata |
| `stat.npy` | ROI definitions (coordinates, shape stats) |
| `F.npy` | Raw fluorescence traces (n_rois × n_frames) |
| `Fneu.npy` | Neuropil fluorescence traces |
| `spks.npy` | Deconvolved spike estimates |
| `iscell.npy` | Cell classification [is_cell, probability] |
| `data.bin` | Registered movie (if `keep_reg=True`) |

### Additional Outputs

| File | Description |
|------|-------------|
| `max_projection_image.png` | Maximum intensity projection |
| `mean_image.png` | Temporal mean image |
| `segmentation_*.png` | ROI masks overlaid on reference image |
| `traces_*.png` | Sample fluorescence traces |
| `pc_metrics/` | Registration quality metrics (if >1500 frames) |

---

## 5. Load and Analyze Results

In [ ]:
# Find ops files
ops_files = mbo.get_files(save_path, str_contains="ops.npy", max_depth=3)
print(f"Found {len(ops_files)} processed planes")

# Load results from first plane
if ops_files:
    results = lsp.load_planar_results(ops_files[0])

    print(f"\nPlane results:")
    print(f"  Total ROIs: {len(results['stat'])}")
    print(f"  Accepted cells: {results['iscell'].sum()}")
    print(f"  Frames: {results['F'].shape[1]}")
    print(f"  F shape: {results['F'].shape}")

### Calculate ΔF/F

In [ ]:
# Calculate ΔF/F with rolling percentile baseline
if ops_files:
    dff = lsp.dff_rolling_percentile(
        results['F'],
        window_size=300,    # frames (~10× tau × fs)
        percentile=20       # baseline percentile
    )

    # Filter for accepted cells only
    dff_cells = dff[results['iscell']]
    print(f"ΔF/F shape (accepted cells): {dff_cells.shape}")

---

## 6. Open Suite2p GUI

The Suite2p GUI provides interactive visualization and manual curation:

- **View registered movie**: Suite2p → Registration → View Registration Binary
- **Compare raw vs registered**: Check "View raw binary" (requires `keep_raw=True`)
- **Registration quality**: Suite2p → Registration → View Registration Metrics (>1500 frames)

In [ ]:
# Open GUI for manual curation
if ops_files:
    stat_file = ops_files[0].parent / "stat.npy"
    if stat_file.exists():
        from suite2p import gui
        gui.run(statfile=str(stat_file))

---

## 7. Volumetric Analysis

When processing multiple z-planes with `run_volume()`, the pipeline automatically generates publication-quality figures for comprehensive volumetric analysis. You can also generate these manually:

In [ ]:
# Load data from all planes for volumetric analysis
all_results = []
for ops_file in ops_files:
    try:
        res = lsp.load_planar_results(ops_file)
        all_results.append(res)
    except Exception as e:
        print(f"Error loading {ops_file}: {e}")

if len(all_results) > 1:
    # Consolidate all planes
    all_stat = np.concatenate([res["stat"] for res in all_results])
    all_iscell = np.vstack([res["iscell"] for res in all_results])
    all_F = np.concatenate([res["F"] for res in all_results], axis=0)
    all_Fneu = np.concatenate([res["Fneu"] for res in all_results], axis=0)

    print(f"Consolidated volume:")
    print(f"  Total ROIs: {len(all_stat)}")
    print(f"  Accepted: {all_iscell[:, 0].sum()}")
    print(f"  Planes: {len(all_results)}")
    print(f"  F shape: {all_F.shape}")

### Multi-Plane Mask Overview

Visualize detected ROIs across all planes in a single figure:

In [ ]:
# Plot masks from all planes in a grid
if len(all_results) > 1:
    fig = lsp.plot_multiplane_masks(
        suite2p_path=save_path,
        stat=all_stat,
        iscell=all_iscell,
        nrows=3,
        ncols=5,
        save_path=save_path / "all_planes_masks.png"  # Optional: save to file
    )

### Volume Quality Metrics

Generate publication-quality figures showing ROI quality across all planes:

In [ ]:
# Quality metrics: ROI counts, compactness, size distributions
if len(all_results) > 1:
    fig = lsp.plot_plane_quality_metrics(
        stat=all_stat,
        iscell=all_iscell,
        save_path=save_path / "volume_quality_metrics.png",
        style="publication"  # or "dark" for dark background
    )

### Trace Analysis

Comprehensive analysis of fluorescence traces across the volume:

In [ ]:
# Trace analysis: SNR, correlation, activity heatmap
if len(all_results) > 1:
    first_ops = lsp.load_ops(ops_files[0])

    fig, metrics = lsp.plot_trace_analysis(
        F=all_F,
        Fneu=all_Fneu,
        stat=all_stat,
        iscell=all_iscell,
        ops=first_ops,
        save_path=save_path / "volume_trace_analysis.png"
    )

    # Access computed metrics
    print(f"Mean SNR across volume: {np.mean(metrics['snr']):.2f}")
    print(f"Cells with SNR > 2: {np.sum(metrics['snr'] > 2)} ({100*np.mean(metrics['snr'] > 2):.1f}%)")

### Summary Statistics Table

Generate a comprehensive summary table for all planes:

In [ ]:
# Create summary table with per-plane statistics
if len(all_results) > 1:
    summary_df = lsp.create_volume_summary_table(
        stat=all_stat,
        iscell=all_iscell,
        F=all_F,
        Fneu=all_Fneu,
        ops=first_ops,
        save_path=save_path / "volume_summary.csv"
    )

    # Display the table
    print(summary_df.to_string(index=False))

### Automatic Volumetric Outputs

When using `run_volume()`, these figures are automatically generated in the output directory:

| File | Description |
|------|-------------|
| `all_planes_masks.png` | Grid showing ROIs overlaid on mean images for all planes |
| `volume_quality_metrics.png` | ROI counts, compactness, size, and acceptance rates per plane |
| `volume_trace_analysis.png` | SNR distributions, example traces, correlation matrix, activity heatmap |
| `volume_summary.csv` | Per-plane statistics table (ROIs, SNR, acceptance rate) |
| `mean_volume_signal.png` | Mean signal intensity across z-depth |
| `rastermap.png` | Activity sorted by similarity (requires rastermap package) |

---

## Next Steps

- **Parameter tuning**: See [anatomical_grid_search.ipynb](./anatomical_grid_search.ipynb) for systematic parameter optimization
- **Spike inference**: See [tau_spike_inference_analysis.ipynb](./tau_spike_inference_analysis.ipynb) for tau parameter analysis
- **Volume consolidation**: Use `lsp.consolidate_volume()` to merge results from all planes into a single directory
- **Documentation**: See the [User Guide](https://millerbrainobservatory.github.io/LBM-Suite2p-Python/user_guide.html) for detailed parameter explanations